# Выделение групп пользователей на основе поведения

Цель - определить группы пользователей, у которых выше удержание в мобильном приложении "Ненужные вещи" и выше конверсия в целевое ("contacts_show") действие.

Задачи:
- выгрузка данных;
- предобработка данных;
- анализ общего поведения пользователей;
- сегментирование пользователей;
- проверка статистических гипотез.

Для анализа предоставлен датасет с двумя таблицами, которые содержат данные о идентификаторах пользователей, источниках привлечения пользователей, действия, совершенного в приложении, времени, когда действие совершено.

In [1]:
import pandas as pd
from datetime import datetime, timedelta
from matplotlib import pyplot as plt
import seaborn as sns
from scipy import stats as st
import math as mth

ModuleNotFoundError: No module named 'pandas'

## Выгрузка данных

In [ ]:
mob_sources = pd.read_csv('https://code.s3.yandex.net/datasets/mobile_sources.csv')

In [ ]:
display(mob_sources.head(10))
print(mob_sources.info())

В таблице с источником два столбца, 4293 строки. Пропусков нет.

In [ ]:
mob_dataset = pd.read_csv('https://code.s3.yandex.net/datasets/mobile_dataset.csv')

display(mob_dataset.head(10))
print(mob_dataset.info())

В таблице с событиями три столбца, 74197 строки. Пропусков данных нет.

## Предобработка данных

In [ ]:
# Изменение типа данных для столбца с датой и временем
mob_dataset['event.time'] = pd.to_datetime(mob_dataset['event.time'])

In [ ]:
# Изменение названий столбцов для удобства
mob_dataset = mob_dataset.rename(columns={'event.time':'event_time', 'event.name':'event_name', 'user.id':'user_id'})
mob_sources = mob_sources.rename(columns={'userId':'user_id'})

In [ ]:
mob_sources.duplicated().sum()

In [ ]:
mob_dataset.duplicated().sum()

In [ ]:
mob_dataset.loc[mob_dataset['event_name']=='show_contacts', 'event_name'] = 'contacts_show' # замена неявных дубликатов,
                                                                                            # указанных в задании

В таблицах не встречены явные дубликаты.

## Анализ общего поведения данных

### Расчет Retention Rate

In [ ]:
dt = mob_dataset.sort_values(['user_id', 'event_time'])

In [ ]:
dt['event_time'] = pd.to_datetime(dt['event_time'])

Окончанием сессии принимается действие пользователя, когда до следующего действия проходит более 30 минут. С одной стороны меньшее время указывает на более частое вовлечение пользователя в приложение, это характерно обычно для приложений социальных сетей или мессенджеров. С другой стороны - большее время характерно для других типов приложений (онлайн банк, услуги доставки еды и т.д.). За время 30 минут, если это было случайное прерывание сессии можно успеть ответить на телефонный звонок, сообщение и т.п. А уже свыше 30 минут может означать, что пользователь уже перестроил свои мысли в другом направлении, вряд ли он вернется к тому же товару спустя такое время.

In [ ]:
# деление событий по сессиям, принято, что между сессиями более 30 минут 
diff_etime = dt.groupby('user_id')['event_time'].diff()
new_session = (diff_etime.isnull()) | (diff_etime > pd.Timedelta('30Min'))
dt['session_id'] = dt.loc[new_session, ['user_id', 'event_time']].groupby('user_id').rank(method='first').astype(int)
dt['session_id'] = dt['session_id'].fillna(method='ffill').astype(int)

In [ ]:
# определение функции для формирования таблицы с показателями удержания до 7 дня
def retention_calc(data):
    dt1 = data.sort_values('event_time')
    dt1.event_time = dt1.event_time.dt.date
    
    df_cohort = dt1.groupby('user_id')['event_time'].min().reset_index().rename(columns={'event_time':'cohort_date'})

    df = dt1.merge(df_cohort, on='user_id')
    
    df["days_diff"] = (df["event_time"] - df["cohort_date"]).dt.days

    cohort_sizes = df_cohort.groupby("cohort_date")["user_id"].nunique().reset_index()
    cohort_sizes.columns = ["cohort_date", "cohort_size"]

    retention_data = df.groupby(["cohort_date", "days_diff"])["user_id"].nunique().reset_index()
    retention_data = retention_data.pivot_table(
    index="cohort_date",
    columns="days_diff",
    values="user_id",
    )

    retention_data = retention_data.merge(cohort_sizes, on="cohort_date")

    for day in range(1,8):
        if day in retention_data.columns:
            retention_data[f"день_{day}"] = (retention_data[day] / retention_data["cohort_size"]) * 100
        
    retention_rate = retention_data[["cohort_date", "cohort_size"] + [f"день_{day}" for day in range(1,8) if f"день_{day}" in retention_data.columns]]
    
    return retention_rate

In [ ]:
retention_rate1 = retention_calc(mob_dataset)

plt.figure(figsize = (15, 6))
sns.heatmap(
    retention_rate1.drop(columns = ['cohort_size']).set_index(['cohort_date']),
    annot = True, 
    fmt = '.2f',
) 
plt.title('Тепловая карта Retention Rate до 7 дня')
plt.ylabel('Дата старта когорты')
plt.show()

По тепловой карте заметно, что в данных представлена информация о юзерах, начинавших использовать приложение с 7 октября по 3 ноября 2019 года.

К седьмому дню удержание в основном превышает 5%, самое низкое для когорты, начавшейся 24 октября (4.32%), самое высокое - для когорты 21 октября (10.56%).

### Определение времени, проведенного в приложении пользователем

In [ ]:
# определение функции, делящей на группы по уровню продолжительности использования приложения
def time_group(x):
    if x < 0.25:
        return 'низкая'
    elif x < 1:
        return 'средняя'
    else:
        return 'высокая'

In [ ]:
session_start = dt.groupby(['user_id', 'session_id'])['event_time'].min().reset_index().rename(columns={'event_time':'start_time'})
session_end = dt.groupby(['user_id', 'session_id'])['event_time'].max().reset_index().rename(columns={'event_time':'end_time'})
sessions = session_start.merge(session_end, on=['user_id', 'session_id'])
sessions['session_duration'] = (sessions['end_time'] - sessions['start_time'])
full_time = sessions.groupby(['user_id'])['session_duration'].sum().reset_index()
full_time['session_duration'] = round(full_time['session_duration'].dt.total_seconds() / 3600, 2)

full_time['time_group'] = full_time.session_duration.apply(time_group)

full_time

In [ ]:
display(full_time.groupby(['time_group'])['user_id'].count().reset_index())

Принимая, что низкая продолжительность использования приложения менее 15 минут, средняя - менее 60 минут, а высокая - не менее 60 минут, получились следующие группы:
- с низкой продолжительностью 2117 пользователей;
- со средней продолжительностью 1669 пользователей;
- с высокой продолжительностью 507 пользователей.

### Определение частоты действий пользователей

In [ ]:
# определение функции, делящей на группы по частоте совершаемых событий
def ev_p_d(x):
    if x < 5:
        return 'низкая'
    elif x < 10:
        return 'средняя'
    else:
        return 'высокая'

In [ ]:
event_freak = dt.groupby(['user_id']).agg(event_cnt=('event_name','count'), first_event=('event_time','min'),
                                                   last_event=('event_time','max')).reset_index()
event_freak['full_time'] = (event_freak.last_event - event_freak.first_event).dt.days + 1

event_freak['events_per_day'] = round(event_freak.event_cnt / event_freak.full_time, 1)

event_freak['events_per_day_group'] = event_freak['events_per_day'].apply(ev_p_d)

event_freak

In [ ]:
display(event_freak.groupby(['events_per_day_group'])['user_id'].count().reset_index())

Принимая, что низкая частота действий менее 5 действий, средняя - менее 10 действий, а высокая - не менее 10 действий, получились следующие группы:
- с низкой частотой 1558 пользователей;
- со средней частотой 1433 пользователя;
- с высокой частотой 1302 пользователя.

### Определение конверсии в целевое действие (просмотр контактов)

In [ ]:
# определение функции, рассчитывающей конверсию в целевое событие по источникам привлечения пользователей,
# а так же общую конверсию
def conv_final_event(data):

    user_events = data.groupby(['user_id'])['event_name'].apply(list).reset_index()

    def check_conv(x):
        return 'contacts_show' in x

    user_events['user_conversion'] = user_events['event_name'].apply(check_conv)

    user_events = user_events.merge(mob_sources[['user_id','source']], on='user_id', how='left')

    conv_by_source = user_events.groupby('source')['user_conversion'].agg(
        total_users='count',
        converted_users='sum',
        conversion_rate='mean'
    ).reset_index()

    return conv_by_source, user_events['user_conversion'].mean()

In [ ]:
display(conv_final_event(dt)[0])
print('Полная конверсия -', round(conv_final_event(dt)[1],2))

Наиболее конвертируемые пользователи привлечены из источника "Yandex", конверсия около 25%. Конверсия пользователей из "Google" чуть меньше - около 24%. Из других источников пользователи конвертируются хуже - около 19%.

## Сегментирование пользователей

Проверка будет происходить по разделению пользователей по частоте действий в приложении. Как указывалось выше, выделены три группы: с низкой частотой (менее 5 действий в день), со средней частотой (менее 10 действий в день) и с высокой частотой (не менее 10 действий в день).

Частота совершения действий - хороший критерий для деления пользователей на группы, потому что:
- Частота отражает вовлечённость. В теории пользователи с высокой частотой действий обычно более лояльны и склонны к конверсии;
- Частота может быть связана с бизнес-показателями (конверсией в покупку);
- Пользователи, которые часто производят действия в теории должны быть вовлечены в процесс и открывать приложение (удерживаться) лучше, чем те, кто редко производит действия.

### Определение лучших по Retention Rate

In [ ]:
low_events = event_freak.loc[event_freak['events_per_day_group']=='низкая', ['user_id',]].reset_index()#.merge(mob_dataset[''])
middle_events = event_freak.loc[event_freak['events_per_day_group']=='средняя', ['user_id',]].reset_index()
high_events = event_freak.loc[event_freak['events_per_day_group']=='высокая', ['user_id',]].reset_index()

In [ ]:
low_events = low_events.merge(mob_dataset[['user_id','event_time','event_name']], on=['user_id'], how='left')

retention_rate_low = retention_calc(low_events)

plt.figure(figsize = (15, 6))
sns.heatmap(
    retention_rate_low.drop(columns = ['cohort_size']).set_index(['cohort_date']),
    annot = True, 
    fmt = '.2f',
) 
plt.title('Тепловая карта Retention Rate до 7 дня для когорты с низкой частотой событий')
plt.ylabel('Дата старта когорты')
plt.show()

In [ ]:
middle_events = middle_events.merge(mob_dataset[['user_id','event_time','event_name']], on=['user_id'], how='left')

retention_rate_middle = retention_calc(middle_events)

plt.figure(figsize = (15, 6))
sns.heatmap(
    retention_rate_middle.drop(columns = ['cohort_size']).set_index(['cohort_date']),
    annot = True, 
    fmt = '.2f',
) 
plt.title('Тепловая карта Retention Rate до 7 днядля когорты со средней частотой событий')
plt.ylabel('Дата старта когорты')
plt.show()

In [ ]:
high_events = high_events.merge(mob_dataset[['user_id','event_time','event_name']], on=['user_id'], how='left')

retention_rate_high = retention_calc(high_events)

plt.figure(figsize = (15, 6))
sns.heatmap(
    retention_rate_high.drop(columns = ['cohort_size']).set_index(['cohort_date']),
    annot = True, 
    fmt = '.2f',
) 
plt.title('Тепловая карта Retention Rate до 7 дня для когорты с высокой частотой событий')
plt.ylabel('Дата старта когорты')
plt.show()

Сравнивая тепловые карты для разных групп, можно прийти к выводу, что наиболее удерживаемые пользователи с низкой частотой действия, а наименее удерживаемые - с высокой. То есть чем меньше в день пользователь совершает действий в приложении, тем чаще в последующие дни он обращается к приложению.

Это не совпадает с предполагаемыми выводами. Нужны будут дополнительные исследования.

### Определение лучших по конверсии в целевое действие

In [ ]:
display(conv_final_event(high_events)[0])
print('Полная конверсия когорты с высокой частотой событий -', round(conv_final_event(high_events)[1],2))

In [ ]:
display(conv_final_event(low_events)[0])
print('Полная конверсия когорты с низкой частотой событий -', round(conv_final_event(low_events)[1],2))

In [ ]:
display(conv_final_event(middle_events)[0])
print('Полная конверсия когорты со средней частотой событий -', round(conv_final_event(middle_events)[1],2))

Наибольшая конверсия в целевое действие наблюдается для пользователей, которые совершают наибольшее количество действий в день - около 26%. Наименьшая конверсия - для пользователей с со средней частотой действий (от 5 до 10 в день) - около 20%.

Эти результаты совпадают с предполагаемыми выводами.

## Проверка статистических гипотез

### Некоторые пользователи установили приложение по ссылке из yandex, другие — из google . Гипотеза: две эти группы демонстрируют разную конверсию в просмотры контактов

Нулевая гипотеза: группы пользователей, привлеченных из Google и Yandex, демонстрируют одинаковую конверсию в просмотры контактов. Альтернативная гипотеза: пользователи, привлеченные из разных источников, демонстрируют различную конверсию в просмотры контактов.

In [ ]:
hip_1 = conv_final_event(dt)[0]
hip_1 = hip_1[hip_1['source'].isin(['google','yandex'])]

n_google = hip_1.loc[hip_1['source']=='google', ['total_users']].values[0]
conv_google = hip_1.loc[hip_1['source']=='google', ['converted_users']].values[0]
p_google = hip_1.loc[hip_1['source']=='google', ['conversion_rate']].values[0]

n_yandex = hip_1.loc[hip_1['source']=='yandex', ['total_users']].values[0]
conv_yandex = hip_1.loc[hip_1['source']=='yandex', ['converted_users']].values[0]
p_yandex = hip_1.loc[hip_1['source']=='yandex', ['conversion_rate']].values[0]

p_combined = (conv_google + conv_yandex)/(n_google+n_yandex)

z_value = (p_google-p_yandex)/mth.sqrt(p_combined * (1-p_combined) * (1/n_google + 1/n_yandex))

distr = st.norm(0,1)

p_value = (1 - distr.cdf(abs(z_value))) * 2

print('p-value:', round(*p_value, 3))

if p_value < 0.05:
    print('Группы из разных источников демонстрируют разную конверсию')
else:
    print('Группы из разных источников демонстрируют схожую конверсию')

### H₀: Величина конверсии в целевое действие не зависит от времени сессии (утро/вечер). H₁:  Величина конверсии в целевое действие выше в вечерние сессии, чем в утренние

За утро примем интервал суток от 4:00 до 10:00, а вечер - с 18:00 до 24:00.

In [ ]:
mor_ev_ses = dt.groupby(['user_id','session_id'])['event_time'].agg(start_ses='min', finish_ses='max',).reset_index()

morning_sessions = mor_ev_ses[(mor_ev_ses['start_ses'].dt.hour >= 4) & (mor_ev_ses['finish_ses'].dt.hour < 10)].merge(dt[['user_id','session_id','event_name']],on=['user_id','session_id'],how='left')

evening_sessions = mor_ev_ses[(mor_ev_ses['start_ses'].dt.hour >= 18)].merge(dt[['user_id','session_id','event_name']],on=['user_id','session_id'],how='left')

In [ ]:
conv_morning = conv_final_event(morning_sessions)[0]
conv_evening = conv_final_event(evening_sessions)[0]

In [ ]:
n_morning = conv_morning['total_users'].sum()
converted_morning = conv_morning['converted_users'].sum()
p_morning = converted_morning / n_morning

n_evening = conv_evening['total_users'].sum()
converted_evening = conv_evening['converted_users'].sum()
p_evening = converted_evening / n_evening

p_combined = (converted_morning + converted_evening)/(n_morning+n_evening)

z_value = (p_morning-p_evening)/mth.sqrt(p_combined * (1-p_combined) * (1/n_morning + 1/n_evening))

distr = st.norm(0,1)

p_value = (1 - distr.cdf(abs(z_value))) * 2

print('p-value:', round(p_value, 3))

if p_value < 0.05:
    print('Конверсии утренних и вечерних сессий различаются')
    print('Конверсия утром -', round(p_morning,3), '\nКонверсия вечером -', round(p_evening,3))
else:
    print('Конверсии утренних и вечерних сессий одинаковые')

Конверсия в целевое действие, наблюдаемая в вечерние сессии, выше, чем в утренние сессии. В вечерних сессиях конверсия выше (18,4% против 14,9%). Стоит отметить одно допущение, что часть пользователей попала в обе группы.

## Выводы

Были изучены данные о действиях пользователей в мобильном приложении "Ненужные вещи". Определено, что показатель удержания на седьмой день после начала использования почти для всех когорт превышает 5%. Конверсии пользователей в целевое действие (contacts_show), привлеченных из Google и Yandex, примерно равны (24-25%), из других источников ниже (около 19%). 

Исследуя удержание пользователей и конверсию в целевое действие групп, разделенных по частоте совершения событий, констатируем, что удерживается лучше группа, у которой частота совершения действий низкая, а конверсия в целевое действие выше у группы с высокой частотой действия.

Проверяя гипотезу о том, что две группы пользователей, привлеченных из Google и Yandex, демонстрируют разную конверсию в просмотры контактов, приходим к выводу, что нет оснований отвергнуть нулевую гипотезу (p=0.82).

Проверяя другую гипотезу, что величина конверсии в целевое действие не зависит от времени сессии (утро/вечер), пришли к выводу, что гипотеза не верна, конверсии разные. Конверсия в вечерние сессии выше, чем в утренние.

Возможные рекомендации:
- обратить внимание на длительность использования мобильного приложения юзерами. С чем может быть связана такая разница между группами пользователей с высокой длительностью и низкой длительностью;
- обратить внимание на группы пользователей, поделенные по частоте совершения действий. С чем может быть связано то, что наиболее активные пользователи удерживаются хуже, чем те, что совершают действия в приложении реже. Либо юзеры быстро находят то, что их интересует, либо их со временем перестает привлекать приложение. Возможно необходимо настроить push-уведомления, чтобы пользователи заходили чаще;
- чтобы улучшить конверсию в утреннее время, нужно внедрить какие-то алгоритмы подбора товаров, которые могут быть интересны людям утром, на что они сами не обращают внимание, это может улучшить показатели конверсии.